# Transformers Error Analysis

### 1. Add imports

In [2]:
import pandas as pd 
import torch 

from torch.utils.data import Dataset, DataLoader
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, DataCollatorWithPadding)
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [3]:
df_test = pd.read_csv("../datasets/clinical_cases_test.csv")

X_test = df_test["medical_abstract"].tolist()

y_test = [
    label - 1 
    for label in df_test["condition_label"].tolist()
]

print(f"Test samples: {len(df_test)}")

Test samples: 2888


In [4]:
MODEL_PATH = "../models/biomedbert-medintake"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_PATH)

device = torch.device(
    "mps" if torch.backends.mps.is_available() else "cpu"
)

model.to(device)

print(f"Device: {device}")


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Device: mps


In [5]:
test_encodings = tokenizer(
    X_test,
    trunction=True,
    max_length=256
)

print("Tokenization completed.")

Tokenization completed.


In [6]:
data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

In [7]:
class MedicalDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {
            key: torch.tensor(val[idx])
            for key, val in self.encodings.items()
        }

        item["labels"] = torch.tensor(self.labels[idx])

        return item


test_dataset = MedicalDataset(test_encodings, y_test)

print(f"Test samples: {len(test_dataset)}")

Test samples: 2888


In [8]:
test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    collate_fn=data_collator
)

print(f"Batches: {len(test_loader)}")

Batches: 361


In [9]:
model.eval()

all_predictions = []
all_labels = []

with torch.no_grad():
    for batch in test_loader:
        batch = {
            key: value.to(device)
            for key, value in batch.items()
        }

        outputs = model(**batch)

        predictions = torch.argmax(outputs.logits, dim=1)

        all_predictions.extend(predictions.cpu().tolist())
        all_labels.extend(batch["labels"].cpu().tolist())

print(f"Predictions: {len(all_predictions)}")
print(f"Labels: {len(all_labels)}")


Predictions: 2888
Labels: 2888


In [10]:
errors = df_test.copy()

errors["true_label"] = all_labels
errors["predicted_label"] = all_predictions

errors = errors[
    errors["true_label"] != errors["predicted_label"]
]

print(f"Errors: {len(errors)}")

Errors: 1000


In [18]:
label_names = {
    0: "(1) Neoplasms",
    1: "(2) Digestive system diseases",
    2: "(3) Nervous system diseases",
    3: "(4) Cardiovascular diseases",
    4: "(5) General pathological conditions"
}

error_table = (
    errors
    .groupby(["true_label", "predicted_label"])
    .size()
    .reset_index(name="errors")
    .sort_values("errors", ascending=False)
    .reset_index(drop=True)
)

error_table["true_label"] = error_table["true_label"].map(label_names)
error_table["predicted_label"] = error_table["predicted_label"].map(label_names)

error_table = error_table.rename(columns={
    "true_label": "True class",
    "predicted_label": "Predicted class"
})

error_table

,True class,Predicted class,errors
0,(5) General pathological conditions,(4) Cardiovascular diseases,199
1,(5) General pathological conditions,(1) Neoplasms,161
2,(5) General pathological conditions,(2) Digestive system diseases,136
3,(5) General pathological conditions,(3) Nervous system diseases,99
4,(4) Cardiovascular diseases,(5) General pathological conditions,58
5,(3) Nervous system diseases,(5) General pathological conditions,55
6,(2) Digestive system diseases,(1) Neoplasms,44
7,(2) Digestive system diseases,(5) General pathological conditions,37
8,(3) Nervous system diseases,(1) Neoplasms,33
9,(3) Nervous system diseases,(4) Cardiovascular diseases,31


In [20]:
errors_5_to_4 = errors[
    (errors["true_label"] == 4) &
    (errors["predicted_label"] == 3)
]

print(f"Errors (5) --> (4): {len(errors_5_to_4)}")

# visualizing full text rows to get better reasoning of vectorizer errors 
pd.set_option("display.max_colwidth", None)

errors_5_to_4[
    ["medical_abstract", "true_label", "predicted_label"]
].head(10)

Errors (5) --> (4): 199


,medical_abstract,true_label,predicted_label
35,"Epidural anaesthesia for labour and caesarean section in a parturient with a single ventricle and transposition of the great arteries. We describe a case of a 29-year-old parturient with a single ventricle and transposition of the great arteries who had lumbar epidural analgesia/anaesthesia with a local anaesthetic for labour, emergency Caesarean section and postoperative pain. Her outcome and that of her baby was successful. The anaesthetic techniques used in other parturients with similar congenital cardiac anomalies are reviewed.",4,3
50,Aberrant origin of the right coronary artery as a potential cause of sudden death: successful anatomical correction. A man with an aberrant right coronary artery and haemodynamically important prolapse of the mitral valve was successfully resuscitated. The aberrant right coronary artery was thought to be a possible cause of the cardiopulmonary arrest in this patient. Both lesions were corrected at a single operation.,4,3
75,Catheterization of coronary artery bypass graft from the descending aorta. The increasing frequency of reoperation for coronary artery disease has led to the use of a variety of grafts. This report describes the catheter technique for selective opacification of a saphenous vein graft from the descending thoracic aorta to the posterior coronary circulation.,4,3
91,"Severe hypertension after liver transplantation in alpha 1 antitrypsin deficiency. Five children with alpha 1 antitrypsin deficiency and terminal liver disease received liver grafts; all five became hypertensive and four developed hypertensive encephalopathy. There was evidence of renal disease preoperatively and renal biopsy specimens showed variable glomerulonephritic histology with IgA nephropathy in one, mesangial-proliferative changes in two, and mesangio-capillary glomerulonephritis type I in two. Four hypertensive episodes were preceded by a fall in creatinine clearance. The association of glomerulonephritis with alpha 1 antitrypsin deficiency in children is more common than has been recognised. Affected patients are prone to severe hypertension of probable renal origin after liver transplantation and the renal lesion may affect long term prognosis.",4,3
94,"Assessing the clinical effectiveness of preventive maneuvers: analytic principles and systematic methods in reviewing evidence and developing clinical practice recommendations. A report by the Canadian Task Force on the Periodic Health Examination. This paper examines a process for evaluating clinical effectiveness and developing recommendations in which systematic methods are used to review evidence from published clinical research and to reach sound conclusions about appropriate medical policy. The methodology addresses four important components of the analytic process: (1) the criteria that must be satisfied for a clinical practice to be considered effective; (2) proper methods for reviewing evidence from published clinical research to determine whether a clinical practice meets these criteria (including methods for performing comprehensive literature reviews, for judging the quality of individual studies, and for synthesizing or pooling the results of multiple studies); (3) theoretical and practical concerns in translating the results of the scientific review into sound clinical practice recommendations; and (4) the importance of documentation, guidelines, and other safeguards to minimize the effect the reviewers themselves have on the objectivity and consistency of the analytic methods.",4,3
98,"Caffeine and cardiac arrhythmias. PURPOSE: To review the evidence supporting the belief that caffeine causes cardiac arrhythmias. DATA SOURCES: Studies published since 1982 identified through computerized searches of MEDLINE, TOXLINE, and Chemical Abstracts and a review of bibliographies of relevant articles on the subject of caffeine and cardiac arrhythmias. STUDY SELECTION: All clinical studies examining